# SHA-256 Virtual ISA

This notebook decompiles SHA-256 into a fixed microcoded virtual ISA.
Hex is treated as display only; the machine is expressed as 32-bit lanes and binary instruction words.


In [ ]:
# Optional installs
# stdlib only; no installs required


In [ ]:

"""
sha256_virtual_isa.py
=====================

Best-effort binary decompilation of SHA-256 into a fixed virtual ISA.

What this does
--------------
1. Treats SHA-256 as a lane machine over 32-bit words.
2. Defines a minimal virtual instruction set for the round topology.
3. Encodes each micro-op as a 64-bit instruction word.
4. Builds the fixed microcode for:
   - message schedule expansion
   - the 64 compression rounds
5. Executes the program and verifies against hashlib.
6. Dumps readable binary listings so hex is no longer the only surface.

This is not claiming the real silicon uses this exact ISA.
This is the tightest faithful software ISA we can justify from the algorithm.

Run:
    python sha256_virtual_isa.py
"""

from __future__ import annotations

import hashlib
from dataclasses import dataclass
from enum import IntEnum
from pathlib import Path
from typing import Dict, List, Tuple

MASK32 = 0xFFFFFFFF

# ============================================================================
# SHA-256 STANDARD CONSTANTS
# ============================================================================

H0_STD = [
    0x6A09E667, 0xBB67AE85, 0x3C6EF372, 0xA54FF53A,
    0x510E527F, 0x9B05688C, 0x1F83D9AB, 0x5BE0CD19,
]

K_STD = [
    0x428A2F98, 0x71374491, 0xB5C0FBCF, 0xE9B5DBA5, 0x3956C25B, 0x59F111F1, 0x923F82A4, 0xAB1C5ED5,
    0xD807AA98, 0x12835B01, 0x243185BE, 0x550C7DC3, 0x72BE5D74, 0x80DEB1FE, 0x9BDC06A7, 0xC19BF174,
    0xE49B69C1, 0xEFBE4786, 0x0FC19DC6, 0x240CA1CC, 0x2DE92C6F, 0x4A7484AA, 0x5CB0A9DC, 0x76F988DA,
    0x983E5152, 0xA831C66D, 0xB00327C8, 0xBF597FC7, 0xC6E00BF3, 0xD5A79147, 0x06CA6351, 0x14292967,
    0x27B70A85, 0x2E1B2138, 0x4D2C6DFC, 0x53380D13, 0x650A7354, 0x766A0ABB, 0x81C2C92E, 0x92722C85,
    0xA2BFE8A1, 0xA81A664B, 0xC24B8B70, 0xC76C51A3, 0xD192E819, 0xD6990624, 0xF40E3585, 0x106AA070,
    0x19A4C116, 0x1E376C08, 0x2748774C, 0x34B0BCB5, 0x391C0CB3, 0x4ED8AA4A, 0x5B9CCA4F, 0x682E6FF3,
    0x748F82EE, 0x78A5636F, 0x84C87814, 0x8CC70208, 0x90BEFFFA, 0xA4506CEB, 0xBEF9A3F7, 0xC67178F2,
]

# ============================================================================
# BASIC WORD OPS
# ============================================================================

def rotr(x: int, n: int) -> int:
    return ((x >> n) | (x << (32 - n))) & MASK32

def ch(x: int, y: int, z: int) -> int:
    return (x & y) ^ (~x & z)

def maj(x: int, y: int, z: int) -> int:
    return (x & y) ^ (x & z) ^ (y & z)

def BSIG0(x: int) -> int:
    return rotr(x, 2) ^ rotr(x, 13) ^ rotr(x, 22)

def BSIG1(x: int) -> int:
    return rotr(x, 6) ^ rotr(x, 11) ^ rotr(x, 25)

def SSIG0(x: int) -> int:
    return rotr(x, 7) ^ rotr(x, 18) ^ (x >> 3)

def SSIG1(x: int) -> int:
    return rotr(x, 17) ^ rotr(x, 19) ^ (x >> 10)

def add32(*xs: int) -> int:
    return sum(xs) & MASK32

def bits32(x: int) -> str:
    return format(x & MASK32, "032b")

# ============================================================================
# VIRTUAL ISA
# ============================================================================

class OP(IntEnum):
    NOP     = 0x00
    MOV     = 0x01
    LOADW   = 0x02   # dst <- W[imm]
    LOADK   = 0x03   # dst <- K[imm]
    SSIG0   = 0x04   # dst <- ssig0(src0)
    SSIG1   = 0x05   # dst <- ssig1(src0)
    BSIG0   = 0x06   # dst <- bsig0(src0)
    BSIG1   = 0x07   # dst <- bsig1(src0)
    CH      = 0x08   # dst <- ch(src0,src1,src2)
    MAJ     = 0x09   # dst <- maj(src0,src1,src2)
    ADD2    = 0x0A   # dst <- src0 + src1
    ADD4    = 0x0B   # dst <- src0 + src1 + src2 + src3
    ADD5    = 0x0C   # dst <- src0 + src1 + src2 + src3 + IMMREG
    STOREW  = 0x0D   # W[imm] <- src0
    SHIFT8  = 0x0E   # (a..h) <- round shift with new_a,new_e in fixed regs
    FINAL   = 0x0F   # H[i] <- H[i] + state[i]

# Register file ids
REG = {
    # architectural state
    "a": 0, "b": 1, "c": 2, "d": 3, "e": 4, "f": 5, "g": 6, "h": 7,
    # scratch
    "x0": 8, "x1": 9, "x2": 10, "x3": 11, "x4": 12, "x5": 13, "x6": 14, "x7": 15,
    # fixed role scratch
    "s0": 16, "s1": 17, "ch": 18, "maj": 19, "t1": 20, "t2": 21,
    "kw": 22, "ww": 23, "newa": 24, "newe": 25,
}

REG_INV = {v: k for k, v in REG.items()}

@dataclass
class Insn:
    op: int
    dst: int = 0
    s0: int = 0
    s1: int = 0
    s2: int = 0
    s3: int = 0
    imm: int = 0

    def encode_u64(self) -> int:
        # 64-bit packed binary word:
        # [op:8][dst:8][s0:8][s1:8][s2:8][s3:8][imm:16]
        return (
            ((self.op & 0xFF) << 56)
            | ((self.dst & 0xFF) << 48)
            | ((self.s0 & 0xFF) << 40)
            | ((self.s1 & 0xFF) << 32)
            | ((self.s2 & 0xFF) << 24)
            | ((self.s3 & 0xFF) << 16)
            | (self.imm & 0xFFFF)
        )

    def encode_bin(self) -> str:
        return format(self.encode_u64(), "064b")

    def pretty(self) -> str:
        def rn(x: int) -> str:
            return REG_INV.get(x, f"r{x}")
        return (
            f"{OP(self.op).name:<6s} dst={rn(self.dst):<5s} "
            f"s0={rn(self.s0):<5s} s1={rn(self.s1):<5s} "
            f"s2={rn(self.s2):<5s} s3={rn(self.s3):<5s} imm={self.imm}"
        )

# ============================================================================
# MESSAGE PREP
# ============================================================================

def sha256_pad_single_block(msg: bytes) -> bytes:
    if len(msg) > 55:
        raise ValueError("single-block only (<=55 bytes)")
    nbits = len(msg) * 8
    out = msg + b"\x80"
    while len(out) % 64 != 56:
        out += b"\x00"
    out += nbits.to_bytes(8, "big")
    assert len(out) == 64
    return out

def block_words(msg: bytes) -> List[int]:
    block = sha256_pad_single_block(msg)
    return [int.from_bytes(block[i:i+4], "big") for i in range(0, 64, 4)]

# ============================================================================
# MICROCODE BUILDER
# ============================================================================

def build_schedule_microcode() -> List[Insn]:
    code: List[Insn] = []
    # W[16..63] = ssig1(W[t-2]) + W[t-7] + ssig0(W[t-15]) + W[t-16]
    for t in range(16, 64):
        code += [
            Insn(OP.LOADW, REG["x0"], imm=t-2),
            Insn(OP.SSIG1, REG["x1"], REG["x0"]),
            Insn(OP.LOADW, REG["x2"], imm=t-7),
            Insn(OP.LOADW, REG["x3"], imm=t-15),
            Insn(OP.SSIG0, REG["x4"], REG["x3"]),
            Insn(OP.LOADW, REG["x5"], imm=t-16),
            # x6 = x1 + x2 + x4 + x5
            Insn(OP.ADD4, REG["x6"], REG["x1"], REG["x2"], REG["x4"], REG["x5"]),
            Insn(OP.STOREW, 0, REG["x6"], imm=t),
        ]
    return code

def build_round_microcode() -> List[Insn]:
    code: List[Insn] = []
    for r in range(64):
        code += [
            # s1 = BSIG1(e)
            Insn(OP.BSIG1, REG["s1"], REG["e"]),
            # ch = CH(e,f,g)
            Insn(OP.CH, REG["ch"], REG["e"], REG["f"], REG["g"]),
            # kw = K[r]
            Insn(OP.LOADK, REG["kw"], imm=r),
            # ww = W[r]
            Insn(OP.LOADW, REG["ww"], imm=r),
            # x0 = h + s1 + ch + kw
            Insn(OP.ADD4, REG["x0"], REG["h"], REG["s1"], REG["ch"], REG["kw"]),
            # t1 = x0 + ww
            Insn(OP.ADD2, REG["t1"], REG["x0"], REG["ww"]),
            # s0 = BSIG0(a)
            Insn(OP.BSIG0, REG["s0"], REG["a"]),
            # maj = MAJ(a,b,c)
            Insn(OP.MAJ, REG["maj"], REG["a"], REG["b"], REG["c"]),
            # t2 = s0 + maj
            Insn(OP.ADD2, REG["t2"], REG["s0"], REG["maj"]),
            # newa = t1 + t2
            Insn(OP.ADD2, REG["newa"], REG["t1"], REG["t2"]),
            # newe = d + t1
            Insn(OP.ADD2, REG["newe"], REG["d"], REG["t1"]),
            # shift state
            Insn(OP.SHIFT8),
        ]
    return code

# ============================================================================
# EXECUTION ENGINE
# ============================================================================

class VM:
    def __init__(self, msg: bytes):
        self.msg = msg
        self.reg = [0] * 256
        self.W = block_words(msg) + [0] * 48
        self.K = list(K_STD)
        self.H0 = list(H0_STD)
        self.state_trace: List[Dict[str, int]] = []
        # load initial state
        for name in ("a", "b", "c", "d", "e", "f", "g", "h"):
            self.reg[REG[name]] = self.H0[REG[name]]

    def execute(self, code: List[Insn]):
        for ins in code:
            self.step(ins)

    def step(self, ins: Insn):
        r = self.reg
        op = OP(ins.op)

        if op == OP.NOP:
            return
        elif op == OP.MOV:
            r[ins.dst] = r[ins.s0]
        elif op == OP.LOADW:
            r[ins.dst] = self.W[ins.imm]
        elif op == OP.LOADK:
            r[ins.dst] = self.K[ins.imm]
        elif op == OP.SSIG0:
            r[ins.dst] = SSIG0(r[ins.s0])
        elif op == OP.SSIG1:
            r[ins.dst] = SSIG1(r[ins.s0])
        elif op == OP.BSIG0:
            r[ins.dst] = BSIG0(r[ins.s0])
        elif op == OP.BSIG1:
            r[ins.dst] = BSIG1(r[ins.s0])
        elif op == OP.CH:
            r[ins.dst] = ch(r[ins.s0], r[ins.s1], r[ins.s2]) & MASK32
        elif op == OP.MAJ:
            r[ins.dst] = maj(r[ins.s0], r[ins.s1], r[ins.s2]) & MASK32
        elif op == OP.ADD2:
            r[ins.dst] = add32(r[ins.s0], r[ins.s1])
        elif op == OP.ADD4:
            r[ins.dst] = add32(r[ins.s0], r[ins.s1], r[ins.s2], r[ins.s3])
        elif op == OP.STOREW:
            self.W[ins.imm] = r[ins.s0]
        elif op == OP.SHIFT8:
            newa = r[REG["newa"]]
            newe = r[REG["newe"]]
            a, b, c, d, e, f, g, h = [r[REG[x]] for x in ("a","b","c","d","e","f","g","h")]
            r[REG["h"]] = g
            r[REG["g"]] = f
            r[REG["f"]] = e
            r[REG["e"]] = newe
            r[REG["d"]] = c
            r[REG["c"]] = b
            r[REG["b"]] = a
            r[REG["a"]] = newa
            self.state_trace.append({name: r[REG[name]] for name in ("a","b","c","d","e","f","g","h")})
        elif op == OP.FINAL:
            raise NotImplementedError("FINAL not used in streaming mode.")
        else:
            raise ValueError(f"Unknown op {op}")

    def digest_words(self) -> List[int]:
        state = [self.reg[REG[x]] for x in ("a","b","c","d","e","f","g","h")]
        return [add32(h0, s) for h0, s in zip(self.H0, state)]

    def digest_hex(self) -> str:
        return "".join(f"{x:08x}" for x in self.digest_words())

# ============================================================================
# ISA VIEW / REPORTS
# ============================================================================

def isa_summary() -> str:
    lines = []
    lines.append("SHA-256 VIRTUAL ISA")
    lines.append("===================")
    lines.append("")
    lines.append("Architectural state registers:")
    lines.append("  a b c d e f g h   (8 x 32-bit)")
    lines.append("")
    lines.append("Backing memories:")
    lines.append("  W[0..63]          message schedule RAM")
    lines.append("  K[0..63]          round constant ROM")
    lines.append("")
    lines.append("Core round micro-ops:")
    lines.append("  BSIG1(e)")
    lines.append("  CH(e,f,g)")
    lines.append("  LOADK r")
    lines.append("  LOADW r")
    lines.append("  T1 = h + BSIG1(e) + CH(e,f,g) + K[r] + W[r]")
    lines.append("  BSIG0(a)")
    lines.append("  MAJ(a,b,c)")
    lines.append("  T2 = BSIG0(a) + MAJ(a,b,c)")
    lines.append("  newa = T1 + T2")
    lines.append("  newe = d + T1")
    lines.append("  SHIFT8")
    lines.append("")
    lines.append("Interpretation:")
    lines.append("  The ISA is tiny. The hardness is not in a large opcode vocabulary.")
    lines.append("  The hardness is in repeated constrained transport through the same round grammar.")
    return "\n".join(lines)

def binary_dump_value_table(msg: bytes, out_path: Path):
    words = block_words(msg)
    rows = []
    for i, w in enumerate(words):
        rows.append({
            "index": i,
            "hex": f"{w:08x}",
            "bin": bits32(w),
        })
    out_path.write_text(json.dumps(rows, indent=2))

def binary_dump_microcode(code: List[Insn], out_path: Path, limit: int = 96):
    rows = []
    for i, ins in enumerate(code[:limit]):
        rows.append({
            "pc": i,
            "op": OP(ins.op).name,
            "pretty": ins.pretty(),
            "u64_hex": f"{ins.encode_u64():016x}",
            "u64_bin": ins.encode_bin(),
        })
    out_path.write_text(json.dumps(rows, indent=2))

def binary_dump_round_states(vm: VM, out_path: Path, limit: int = 8):
    rows = []
    for r, st in enumerate(vm.state_trace[:limit]):
        rows.append({
            "round": r,
            "hex": {k: f"{v:08x}" for k, v in st.items()},
            "bin": {k: bits32(v) for k, v in st.items()},
        })
    out_path.write_text(json.dumps(rows, indent=2))

# ============================================================================
# MAIN
# ============================================================================

def main():
    msg = b"Dean Kulik / ISA probe / April 2026"

    schedule_code = build_schedule_microcode()
    round_code = build_round_microcode()

    vm = VM(msg)
    vm.execute(schedule_code)
    vm.execute(round_code)

    digest_vm = vm.digest_hex()
    digest_std = hashlib.sha256(msg).hexdigest()

    out_dir = Path("sha256_virtual_isa_outputs")
    out_dir.mkdir(exist_ok=True)

    (out_dir / "isa_summary.txt").write_text(isa_summary())
    binary_dump_value_table(msg, out_dir / "input_block_words_binary.json")
    binary_dump_microcode(round_code, out_dir / "round_microcode_binary.json", limit=96)
    binary_dump_round_states(vm, out_dir / "round_state_binary.json", limit=8)

    print(isa_summary())
    print()
    print("Verification")
    print("------------")
    print("message     :", msg.decode())
    print("digest(vm)  :", digest_vm)
    print("digest(std) :", digest_std)
    print("match       :", digest_vm == digest_std)
    print()
    print("Program sizes")
    print("-------------")
    print("schedule micro-ops :", len(schedule_code))
    print("round micro-ops    :", len(round_code))
    print("total micro-ops    :", len(schedule_code) + len(round_code))
    print()
    print("First 16 round micro-ops")
    print("------------------------")
    for i, ins in enumerate(round_code[:16]):
        print(f"{i:03d}  {ins.pretty():70s}  {ins.encode_bin()}")
    print()
    print(f"Wrote outputs to: {out_dir.resolve()}")

if __name__ == "__main__":
    main()


In [ ]:
main()
